# PySpark — UDFs (User-Defined Functions)

When built-in PySpark functions are not enough, you can write your own **UDF** (User-Defined Function).

| Type | How to define | Performance | When to use |
|------|--------------|-------------|-------------|
| Regular UDF | `@udf(returnType)` | Slow (Python row-by-row) | Custom logic, simple transformations |
| Pandas UDF | `@pandas_udf(returnType)` | Fast (Apache Arrow, vectorized) | Numeric/string operations on Series |

**Key rule:** Always prefer **built-in PySpark functions** first. Use UDFs only when there is no built-in equivalent — built-in functions stay in JVM and never cross to Python.

## Setup

In [ ]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql.functions import udf, pandas_udf, col
from pyspark.sql.types import StringType, IntegerType, DoubleType, ArrayType
import pandas as pd

active = SparkSession.getActiveSession()
if active:
    active.stop()

spark = SparkSession.builder \
    .appName("UDFs") \
    .config("spark.pyspark.python", sys.executable) \
    .getOrCreate()

data = [
    (1, "Alice Smith",   "Engineering", 95000, "Python,SQL,Spark"),
    (2, "Bob Jones",     "Marketing",   72000, "Excel,PowerBI"),
    (3, "Charlie Brown", "Engineering", 110000, "Java,Scala,Spark"),
    (4, "Diana Prince",  "HR",          65000, "HRIS,Excel"),
    (5, "Eve Online",    "Marketing",   85000, "Google Ads,SQL"),
]
df = spark.createDataFrame(data, ["id", "full_name", "dept", "salary", "skills"])
df.show(truncate=False)

## 1. Regular UDF — Row-by-Row Processing

PySpark serialises data to Python, calls your function once per row, then serialises back to JVM.  
Simple to write but **slow for large datasets** — use built-in functions when possible.

In [ ]:
# Method 1: decorator style
@udf(returnType=StringType())
def get_first_name(full_name):
    """Extract first name from 'First Last' format."""
    if full_name is None:
        return None
    return full_name.split()[0]

# Method 2: explicit registration
def salary_grade(salary):
    """Classify salary into a grade band."""
    if salary is None:   return "Unknown"
    if salary >= 100000: return "Senior"
    if salary >= 80000:  return "Mid"
    return "Junior"

grade_udf = udf(salary_grade, StringType())

# Apply UDFs like any built-in function
df.withColumn("first_name", get_first_name(col("full_name"))) \
  .withColumn("grade",      grade_udf(col("salary"))) \
  .select("full_name", "first_name", "salary", "grade") \
  .show()

## 2. UDF That Returns a Complex Type

In [ ]:
# UDF that returns a list (ArrayType)
@udf(returnType=ArrayType(StringType()))
def parse_skills(skills_str):
    """Split comma-separated skills string into a list."""
    if skills_str is None:
        return []
    return [s.strip() for s in skills_str.split(",")]

df_skills = df.withColumn("skills_list", parse_skills(col("skills")))
df_skills.select("full_name", "skills", "skills_list").show(truncate=False)

# Count skills per person using the list
from pyspark.sql.functions import size
df_skills.withColumn("skill_count", size(col("skills_list"))).select("full_name", "skill_count").show()

## 3. Registering UDFs for SQL Use

Register a UDF to make it available inside `spark.sql()` queries.

In [ ]:
# Register the UDF with a SQL name
spark.udf.register("salary_grade_sql", salary_grade, StringType())
spark.udf.register("first_name_sql",   lambda s: s.split()[0] if s else None, StringType())

# Now use in SQL
df.createOrReplaceTempView("employees")
spark.sql("""
    SELECT
        full_name,
        first_name_sql(full_name)  AS first_name,
        salary,
        salary_grade_sql(salary)   AS grade
    FROM employees
    ORDER BY salary DESC
""").show()

## 4. Pandas UDF — Vectorized (Fast)

Pandas UDFs use **Apache Arrow** to transfer data in batches between JVM and Python.  
Instead of calling your function once per row, it receives a `pd.Series` (a whole batch at once).

**Pandas UDFs are 10x–100x faster than regular UDFs for numeric and string operations.**

In [ ]:
import pandas as pd
from pyspark.sql.functions import pandas_udf
from pyspark.sql.types import DoubleType, StringType

# Pandas UDF — receives pd.Series, returns pd.Series
@pandas_udf(DoubleType())
def apply_tax(salary: pd.Series) -> pd.Series:
    """Apply 30% tax — vectorized operation on entire column batch."""
    return salary * 0.70   # after-tax salary

@pandas_udf(StringType())
def grade_pandas(salary: pd.Series) -> pd.Series:
    """Classify salary band — vectorized with pd.cut."""
    return pd.cut(
        salary,
        bins=[0, 80000, 100000, float('inf')],
        labels=["Junior", "Mid", "Senior"]
    ).astype(str)

df.withColumn("net_salary",  apply_tax(col("salary"))) \
  .withColumn("grade",       grade_pandas(col("salary"))) \
  .select("full_name", "salary", "net_salary", "grade") \
  .show()

## 5. Regular UDF vs Pandas UDF — Performance

In [ ]:
import time

# Generate large DataFrame
big_df = spark.range(500_000).toDF("id") \
    .withColumn("salary", (col("id") % 100 * 1000 + 50000).cast("double"))

# Regular UDF
@udf(DoubleType())
def tax_regular(salary):
    return salary * 0.70 if salary else None

# Pandas UDF
@pandas_udf(DoubleType())
def tax_pandas(salary: pd.Series) -> pd.Series:
    return salary * 0.70

# Benchmark
t0 = time.time()
big_df.withColumn("net", tax_regular(col("salary"))).count()
t1 = time.time()
big_df.withColumn("net", tax_pandas(col("salary"))).count()
t2 = time.time()

print(f"Regular UDF  : {t1-t0:.2f}s")
print(f"Pandas UDF   : {t2-t1:.2f}s")
print(f"Speedup      : ~{(t1-t0)/(t2-t1):.1f}x")

## 6. Null Handling in UDFs — Always Check for None

In [ ]:
# Null safety is YOUR responsibility in UDFs — PySpark does not protect you
data_nulls = [(1, "Alice", 95000), (2, None, None), (3, "Charlie", 110000)]
df_nulls = spark.createDataFrame(data_nulls, ["id", "name", "salary"])

# BAD — crashes with AttributeError on None
# @udf(StringType())
# def bad_udf(name):
#     return name.upper()   # None.upper() → AttributeError

# GOOD — always guard against None
@udf(StringType())
def safe_upper(name):
    if name is None:
        return None
    return name.upper()

df_nulls.withColumn("name_upper", safe_upper(col("name"))).show()

## Quick Reference

```python
from pyspark.sql.functions import udf, pandas_udf
from pyspark.sql.types import StringType, IntegerType, DoubleType
import pandas as pd

# Regular UDF — decorator style
@udf(returnType=StringType())
def my_udf(val):
    if val is None: return None
    return val.upper()

# Regular UDF — explicit
my_udf = udf(lambda x: x.upper() if x else None, StringType())

# Register for SQL use
spark.udf.register("my_udf_sql", lambda x: x.upper() if x else None, StringType())

# Apply to DataFrame
df.withColumn("new_col", my_udf(col("existing_col")))

# Pandas UDF — vectorized
@pandas_udf(DoubleType())
def my_pandas_udf(s: pd.Series) -> pd.Series:
    return s * 1.1

df.withColumn("result", my_pandas_udf(col("salary")))

# When to use which:
# Built-in functions    → always first choice (JVM, no Python overhead)
# Regular UDF          → custom logic, low volume
# Pandas UDF           → large volume numeric/string processing
```

## Interview Questions

1. What is a UDF in PySpark? When should you use one?
2. What is the performance cost of a regular UDF? Why is it slower than built-in functions?
3. What is a Pandas UDF and why is it faster than a regular UDF?
4. What is Apache Arrow and how does it help Pandas UDFs?
5. How do you use a UDF in a `spark.sql()` query?
6. What happens if a UDF receives a `null` and you don't handle it?
7. What return type annotation is required for Pandas UDFs?

## Practice Exercises

**Easy:**
1. Write a regular UDF that takes a salary (int) and returns `"High"` if > 90000, else `"Low"`.
2. Write a UDF to capitalise only the first letter of a string (title case).

**Medium:**
3. Write a UDF that returns `True` if a string contains "Spark" (case-insensitive), else `False`.
4. Write a Pandas UDF that normalises a salary column: `(salary - mean) / std`.

**Advanced:**
5. Write a Pandas UDF that reverses each string in a column (vectorized).
6. Register a UDF for SQL use, then query it in a `spark.sql()` CTE.
7. Benchmark your regular UDF vs Pandas UDF vs the built-in equivalent on 1 million rows.

In [ ]:
spark.stop()